In [1]:
import numpy as np
from tqdm import tqdm
import sqlite3
import pandas as pd
from sklearn.model_selection import train_test_split
import os
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

# Load from your local SQLite database
conn = sqlite3.connect("data/complaints.db")
df = pd.read_sql_query("SELECT * FROM complaints", conn)
conn.close()

# Split into train/test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

data = {
    "train": train_df,
    "test": test_df
}


In [2]:
import ollama
import numpy as np
import re
from tqdm import tqdm

y_pred = []
for text in tqdm(data["test"]["complaint_text"]):
    response = ollama.chat(
        model="tinyllama",
        messages=[{
            "role": "user", 
            "content": f"Classify this complaint as positive (1) or negative (0). Answer with ONLY the number 0 or 1, nothing else:\n\n{text}"
        }]
    )
    
    # Extract the first 0 or 1 from the response
    match = re.search(r'[01]', response['message']['content'])
    if match:
        classification = int(match.group())
    else:
        # Default to 0 if we can't parse
        classification = 0
    y_pred.append(classification)

y_pred = np.array(y_pred)


100%|██████████| 100/100 [04:43<00:00,  2.84s/it]


In [ ]:
import ollama
import numpy as np
from tqdm import tqdm

y_pred = []
for text in tqdm(data["test"]["complaint_text"]):
    response = ollama.chat(
        model="phi",  # or your model name qwen3:1.7b
        messages=[{
            "role": "user", 
            "content": f"Classify this complaint as positive (1) or negative (0). Reply with only '0' or '1':\n\n{text}"
        }]
    )
    
    # Parse the response
    classification = int(response['message']['content'].strip())
    y_pred.append(classification)

y_pred = np.array(y_pred)
print(y_pred.mean())

  0%|          | 0/100 [00:04<?, ?it/s]


ValueError: invalid literal for int() with base 10: 'Response:\nBased on the provided text material, "got logged out mid transfer, dont know if it went thru or not, stressful" is a negative complaint. Based on the given material, only \'0\' and \'1\' c

In [3]:
from sklearn.metrics import accuracy_score, classification_report

def evaluate_performance(y_true, y_pred):
    report = classification_report(y_true, y_pred)
    return report

# Convert sentiment_label to binary (positive=1, else=0)
y_true = (data["test"]["sentiment_label"] == "positive").astype(int).values

report = evaluate_performance(y_true, y_pred)
print(report)
print(f"\nAccuracy: {accuracy_score(y_true, y_pred):.2%}")


              precision    recall  f1-score   support

           0       0.78      0.57      0.66        67
           1       0.43      0.67      0.52        33

    accuracy                           0.60       100
   macro avg       0.60      0.62      0.59       100
weighted avg       0.66      0.60      0.61       100


Accuracy: 60.00%


NameError: name 'data' is not defined

NameError: name 'pipe' is not defined